# 02 — Deteksi Wajah, Upper Body, Lower Body & Cropping (InsightFace + YOLOv8n-pose)

Notebook lanjutan `identity-lora-pipeline`: membaca hasil crop tubuh penuh dari
`data/cropped/<identitas>/` (modul 001), mendeteksi wajah dengan InsightFace
`buffalo_sc`, serta upper body dan lower body dari 17 keypoint COCO hasil
YOLOv8n-pose, lalu memotong region masing-masing (dengan padding) ke
`data/face/<identitas>/`, `data/upper_body/<identitas>/`, dan
`data/lower_body/<identitas>/`.

Alur: `data/cropped/<identitas>/` → deteksi wajah (K8) / deteksi pose + kelompok
keypoint (K9-K11) → crop + padding (K14, K15) → tiga folder output (K13). Ketiga
jenis deteksi berjalan independen per gambar (K12): satu gambar bisa menghasilkan
0 sampai 3 crop.

Kelas error dimuat dari `00_pipeline_errors.ipynb`. Sel runner (tag `manual-run`)
memakai hasil crop sungguhan di `data/cropped/` dan hanya menampilkan
`DISPLAY_LIMIT` data pertama.

## 1. Import dan logger

In [ ]:
import logging
import sys
from dataclasses import dataclass, field
from enum import Enum
from itertools import islice
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from insightface.app import FaceAnalysis
from ultralytics import YOLO

# Contoh: 2026-09-26 10:30:00,123 | INFO     | face_upper_lower_body_detection_and_cropping | ...
LOG_FORMAT = "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s"


def setup_logger(name: str) -> logging.Logger:
    """Membuat logger dengan format teks.

    Args:
        name: Nama logger.

    Returns:
        Logger yang siap dipakai.
    """
    created = logging.getLogger(name)
    if created.handlers:
        return created

    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(LOG_FORMAT))
    created.addHandler(handler)
    created.setLevel(logging.INFO)
    return created


logger = setup_logger("face_upper_lower_body_detection_and_cropping")

## 2. Konfigurasi

In [ ]:
# Kernel notebook biasanya berjalan dari folder notebooks/, jadi path relatif "data/raw"
# tidak ketemu. Root project dicari dengan naik dari folder kerja sampai ada data/raw/.
PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "data" / "raw").is_dir()),
    Path.cwd(),
)
CROPPED_DIR = PROJECT_ROOT / "data" / "cropped"  # input, hasil modul 001 (K7)
FACE_DIR = PROJECT_ROOT / "data" / "face"  # output crop wajah (K13)
UPPER_BODY_DIR = PROJECT_ROOT / "data" / "upper_body"  # output crop upper body (K13)
LOWER_BODY_DIR = PROJECT_ROOT / "data" / "lower_body"  # output crop lower body (K13)

FACE_MODEL_NAME = "buffalo_sc"
FACE_DET_THRESH = 0.5  # sementara, belum dikalibrasi pada data asli (K8)

POSE_MODEL_WEIGHTS = "yolov8n-pose.pt"
POSE_CONF_THRESHOLD = 0.5  # ambang deteksi orang, sementara (K9)
KEYPOINT_CONF_THRESHOLD = 0.5  # ambang per-keypoint, terpisah dari POSE_CONF_THRESHOLD (K11)

PADDING_RATIO = 0.1  # padding simetris di semua sisi bounding box sebelum crop (K14)
MIN_SIDE_PX = 64  # sisi terpanjang minimum hasil crop, sementara (K14)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

DISPLAY_LIMIT = 5  # jumlah data pertama yang ditampilkan sel runner

## 3. Error pipeline

In [ ]:
# Definisi error ada di notebook terpisah; %run mengeksekusinya di namespace kernel ini.
%run "{PROJECT_ROOT / 'notebooks' / '00_pipeline_errors.ipynb'}"

## 4. Deteksi wajah (K8)

In [ ]:
@dataclass
class BoundingBox:
    """Satu box hasil deteksi (wajah, atau diturunkan dari keypoint pose)."""

    x1: int
    y1: int
    x2: int
    y2: int
    confidence: float


def load_face_analyzer(model_name: str, det_thresh: float) -> FaceAnalysis:
    """Memuat model deteksi wajah InsightFace.

    Args:
        model_name: Nama model pack InsightFace, mis. "buffalo_sc".
        det_thresh: Ambang confidence minimum deteksi wajah.

    Returns:
        FaceAnalysis yang siap dipakai `detect_faces`, hanya memuat modul deteksi.
    """
    analyzer = FaceAnalysis(name=model_name, allowed_modules=["detection"])
    analyzer.prepare(ctx_id=-1, det_thresh=det_thresh)
    return analyzer


def detect_faces(analyzer: FaceAnalysis, image: np.ndarray) -> list[BoundingBox]:
    """Mendeteksi wajah pada satu gambar.

    Args:
        analyzer: FaceAnalysis hasil `load_face_analyzer`.
        image: Gambar hasil `cv2.imread`.

    Returns:
        Daftar box wajah beserta confidence-nya.

    Raises:
        FaceDetectionError: Kalau model gagal melakukan inferensi pada gambar.
    """
    try:
        faces = analyzer.get(image)
    except (RuntimeError, ValueError, cv2.error) as e:
        logger.error(f"Deteksi wajah gagal: {e}", exc_info=True)
        raise FaceDetectionError(f"Deteksi wajah gagal: {e}") from e
    return [
        BoundingBox(
            x1=int(face.bbox[0]),
            y1=int(face.bbox[1]),
            x2=int(face.bbox[2]),
            y2=int(face.bbox[3]),
            confidence=float(face.det_score),
        )
        for face in faces
    ]


def validate_single_face(boxes: list[BoundingBox]) -> BoundingBox:
    """Memastikan tepat satu wajah pada satu gambar.

    Args:
        boxes: Daftar box hasil `detect_faces`.

    Returns:
        Satu-satunya box wajah yang valid.

    Raises:
        NoFaceDetectedError: Kalau tidak ada wajah sama sekali.
        MultipleFaceDetectedError: Kalau ada lebih dari satu wajah (di luar cakupan MVP).
    """
    if len(boxes) == 0:
        raise NoFaceDetectedError("Tidak ada wajah terdeteksi")
    if len(boxes) > 1:
        raise MultipleFaceDetectedError(
            f"{len(boxes)} wajah terdeteksi, di luar cakupan MVP"
        )
    return boxes[0]

In [ ]:
# Runner "define -> demonstrate": menjalankan detect_faces pada DISPLAY_LIMIT gambar
# pertama di data/cropped/ (hasil modul 001) dan menggambar box wajah hasilnya. Butuh
# data/cropped/<identitas>/ terisi (jalankan notebook 01 lebih dulu) dan koneksi
# internet (untuk mengunduh bobot buffalo_sc saat pertama kali dijalankan) — sengaja
# tidak dieksekusi otomatis saat pemeriksaan kode (lihat README.md); Arya yang
# menjalankannya setelah mengisi data.
_runner_image_paths = list(
    islice(
        (
            image_path
            for identity_dir in sorted(CROPPED_DIR.iterdir())
            if identity_dir.is_dir()
            for image_path in sorted(identity_dir.iterdir())
            if image_path.suffix.lower() in IMAGE_EXTENSIONS
        ),
        DISPLAY_LIMIT,
    )
)

_runner_analyzer = load_face_analyzer(FACE_MODEL_NAME, FACE_DET_THRESH)
_runner_images = [cv2.imread(str(path)) for path in _runner_image_paths]
_runner_faces = [detect_faces(_runner_analyzer, image) for image in _runner_images]

_runner_figure, _runner_axes = plt.subplots(
    1, len(_runner_image_paths), figsize=(4 * len(_runner_image_paths), 6), squeeze=False
)
for _runner_axis, _runner_path, _runner_image, _runner_boxes in zip(
    _runner_axes[0], _runner_image_paths, _runner_images, _runner_faces
):
    _runner_display = _runner_image.copy()
    for _runner_box in _runner_boxes[:DISPLAY_LIMIT]:
        cv2.rectangle(
            _runner_display,
            (_runner_box.x1, _runner_box.y1),
            (_runner_box.x2, _runner_box.y2),
            (0, 255, 0),
            2,
        )
        cv2.putText(
            _runner_display,
            f"{_runner_box.confidence:.2f}",
            (_runner_box.x1, max(_runner_box.y1 - 10, 0)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (0, 255, 0),
            2,
        )
    _runner_axis.imshow(cv2.cvtColor(_runner_display, cv2.COLOR_BGR2RGB))
    _runner_axis.set_title(_runner_path.name, fontsize=8)
    _runner_axis.axis("off")
plt.show()

for _runner_path, _runner_boxes in zip(_runner_image_paths, _runner_faces):
    print(f"{_runner_path.name}: {len(_runner_boxes)} wajah")
    for _runner_box in _runner_boxes[:DISPLAY_LIMIT]:
        print(
            (
                _runner_box.x1,
                _runner_box.y1,
                _runner_box.x2,
                _runner_box.y2,
                _runner_box.confidence,
            )
        )

## 5. Deteksi pose (K9)

In [ ]:
def load_pose_detector(weights: str) -> YOLO:
    """Memuat model YOLOv8n-pose.

    Args:
        weights: Nama atau path bobot model, mis. "yolov8n-pose.pt".

    Returns:
        Model YOLO yang siap dipakai `detect_pose`.
    """
    return YOLO(weights)


def detect_pose(model: YOLO, image_path: Path, conf_threshold: float) -> list[np.ndarray]:
    """Mendeteksi orang dan 17 keypoint COCO pada satu gambar.

    Args:
        model: Model YOLO hasil `load_pose_detector`.
        image_path: Path ke gambar yang akan dideteksi.
        conf_threshold: Ambang confidence minimum deteksi orang.

    Returns:
        Daftar keypoint per orang, tiap elemen berbentuk (17, 3) berisi (x, y, confidence).

    Raises:
        PoseDetectionError: Kalau model gagal melakukan inferensi pada gambar.
    """
    try:
        results = model.predict(
            source=str(image_path),
            conf=conf_threshold,
            verbose=False,
        )
    except (RuntimeError, OSError, cv2.error) as e:
        logger.error(f"Deteksi pose gagal pada {image_path}: {e}", exc_info=True)
        raise PoseDetectionError(f"Deteksi pose gagal pada {image_path}: {e}") from e
    keypoints = results[0].keypoints.data.cpu().numpy()
    return [keypoints[i] for i in range(keypoints.shape[0])]


def validate_single_person_pose(people: list[np.ndarray]) -> np.ndarray:
    """Memastikan tepat satu orang terdeteksi pose pada satu gambar.

    Args:
        people: Daftar keypoint hasil `detect_pose`.

    Returns:
        Keypoint satu-satunya orang yang valid, berbentuk (17, 3).

    Raises:
        NoPersonPoseDetectedError: Kalau tidak ada orang sama sekali.
        MultiplePersonPoseDetectedError: Kalau ada lebih dari satu orang (di luar cakupan MVP).
    """
    if len(people) == 0:
        raise NoPersonPoseDetectedError("Tidak ada orang terdeteksi oleh model pose")
    if len(people) > 1:
        raise MultiplePersonPoseDetectedError(
            f"{len(people)} orang terdeteksi oleh model pose, di luar cakupan MVP"
        )
    return people[0]

In [ ]:
# Runner "define -> demonstrate": menjalankan detect_pose + validate_single_person_pose
# pada gambar yang sama dengan runner deteksi wajah di atas, lalu menggambar titik
# keypoint yang lolos KEYPOINT_CONF_THRESHOLD. Butuh data/cropped/<identitas>/ terisi
# dan koneksi internet (mengunduh bobot yolov8n-pose.pt saat pertama kali dijalankan)
# — sengaja tidak dieksekusi otomatis saat pemeriksaan kode (lihat README.md).
_runner_pose_model = load_pose_detector(POSE_MODEL_WEIGHTS)
_runner_poses_per_image = [
    detect_pose(_runner_pose_model, path, POSE_CONF_THRESHOLD) for path in _runner_image_paths
]

_runner_validated_poses = []
for _runner_path, _runner_image, _runner_people in zip(
    _runner_image_paths, _runner_images, _runner_poses_per_image
):
    try:
        _runner_keypoints = validate_single_person_pose(_runner_people)
    except (NoPersonPoseDetectedError, MultiplePersonPoseDetectedError) as e:
        print(f"{_runner_path.name}: dilewati, {e}")
        continue
    print(f"{_runner_path.name}: lolos, {int((_runner_keypoints[:, 2] >= KEYPOINT_CONF_THRESHOLD).sum())} keypoint lolos ambang")
    _runner_validated_poses.append((_runner_path, _runner_image, _runner_keypoints))

if _runner_validated_poses:
    _runner_figure, _runner_axes = plt.subplots(
        1,
        len(_runner_validated_poses),
        figsize=(4 * len(_runner_validated_poses), 6),
        squeeze=False,
    )
    for _runner_axis, (_runner_path, _runner_image, _runner_keypoints) in zip(
        _runner_axes[0], _runner_validated_poses
    ):
        _runner_display = _runner_image.copy()
        for _runner_x, _runner_y, _runner_conf in _runner_keypoints:
            if _runner_conf >= KEYPOINT_CONF_THRESHOLD:
                cv2.circle(_runner_display, (int(_runner_x), int(_runner_y)), 4, (0, 255, 0), -1)
        _runner_axis.imshow(cv2.cvtColor(_runner_display, cv2.COLOR_BGR2RGB))
        _runner_axis.set_title(_runner_path.name, fontsize=8)
        _runner_axis.axis("off")
    plt.show()

## 6. Kelompok keypoint & bbox (K10, K11)

In [ ]:
# 17 keypoint COCO (indeks 0-16, urutan sesuai output YOLOv8n-pose):
# 0 nose, 1 left_eye, 2 right_eye, 3 left_ear, 4 right_ear, 5 left_shoulder,
# 6 right_shoulder, 7 left_elbow, 8 right_elbow, 9 left_wrist, 10 right_wrist,
# 11 left_hip, 12 right_hip, 13 left_knee, 14 right_knee, 15 left_ankle, 16 right_ankle
UPPER_KEYPOINT_INDICES = set(range(0, 11))
LOWER_KEYPOINT_INDICES = set(range(11, 17))
UPPER_ANCHOR_INDICES = {5, 6}  # left_shoulder, right_shoulder
LOWER_ANCHOR_INDICES = {11, 12}  # left_hip, right_hip


def _points_above_threshold(
    keypoints: np.ndarray, indices: set[int], conf_threshold: float
) -> np.ndarray:
    group = keypoints[sorted(indices)]
    return group[group[:, 2] >= conf_threshold]


def derive_body_part_bbox(
    keypoints: np.ndarray,
    indices: set[int],
    anchor_indices: set[int],
    conf_threshold: float,
) -> BoundingBox:
    """Menurunkan bounding box grup keypoint (upper atau lower body).

    Args:
        keypoints: Keypoint satu orang hasil `validate_single_person_pose`, berbentuk (17, 3).
        indices: Indeks keypoint anggota grup (upper atau lower body).
        anchor_indices: Indeks keypoint anchor grup itu (bahu untuk upper, pinggul untuk lower).
        conf_threshold: Ambang confidence minimum keypoint yang dihitung.

    Returns:
        Bounding box grup, dari titik min/max yang lolos ambang. Confidence-nya
        rata-rata confidence titik yang lolos (hanya untuk log/tampilan).

    Raises:
        InsufficientKeypointsError: Kalau tidak ada anchor yang lolos ambang, atau
            total titik yang lolos di grup itu kurang dari 2.
    """
    anchor_points = _points_above_threshold(keypoints, anchor_indices, conf_threshold)
    group_points = _points_above_threshold(keypoints, indices, conf_threshold)
    if len(anchor_points) == 0 or len(group_points) < 2:
        raise InsufficientKeypointsError(
            f"Keypoint grup tidak cukup: {len(anchor_points)} anchor, "
            f"{len(group_points)} titik total lolos ambang {conf_threshold}"
        )
    return BoundingBox(
        x1=int(group_points[:, 0].min()),
        y1=int(group_points[:, 1].min()),
        x2=int(group_points[:, 0].max()),
        y2=int(group_points[:, 1].max()),
        confidence=float(group_points[:, 2].mean()),
    )

In [ ]:
# Runner "define -> demonstrate": menurunkan bbox upper/lower body dari hasil runner
# validasi pose di atas (_runner_validated_poses), lalu menggambar kotaknya. Sengaja
# tidak dieksekusi otomatis saat pemeriksaan kode (lihat README.md).
_runner_body_part_boxes = []
for _runner_path, _runner_image, _runner_keypoints in _runner_validated_poses:
    _runner_display = _runner_image.copy()
    for _runner_group_name, _runner_indices, _runner_anchor_indices, _runner_color in (
        ("upper", UPPER_KEYPOINT_INDICES, UPPER_ANCHOR_INDICES, (255, 0, 0)),
        ("lower", LOWER_KEYPOINT_INDICES, LOWER_ANCHOR_INDICES, (0, 0, 255)),
    ):
        try:
            _runner_box = derive_body_part_bbox(
                _runner_keypoints, _runner_indices, _runner_anchor_indices, KEYPOINT_CONF_THRESHOLD
            )
        except InsufficientKeypointsError as e:
            print(f"{_runner_path.name} [{_runner_group_name}]: dilewati, {e}")
            continue
        print(
            f"{_runner_path.name} [{_runner_group_name}]: "
            f"{(_runner_box.x1, _runner_box.y1, _runner_box.x2, _runner_box.y2, _runner_box.confidence)}"
        )
        cv2.rectangle(
            _runner_display,
            (_runner_box.x1, _runner_box.y1),
            (_runner_box.x2, _runner_box.y2),
            _runner_color,
            2,
        )
        _runner_body_part_boxes.append((_runner_path, _runner_group_name, _runner_box))
    plt.figure(figsize=(4, 6))
    plt.imshow(cv2.cvtColor(_runner_display, cv2.COLOR_BGR2RGB))
    plt.title(_runner_path.name, fontsize=8)
    plt.axis("off")
    plt.show()

## 7. Cropping (K14, K15)

In [ ]:
def _apply_padding(
    box: BoundingBox, image_shape: tuple[int, int], padding_ratio: float
) -> tuple[int, int, int, int]:
    height, width = image_shape
    box_width = box.x2 - box.x1
    box_height = box.y2 - box.y1
    pad_x = int(box_width * padding_ratio)
    pad_y = int(box_height * padding_ratio)

    x1 = max(box.x1 - pad_x, 0)
    y1 = max(box.y1 - pad_y, 0)
    x2 = min(box.x2 + pad_x, width)
    y2 = min(box.y2 + pad_y, height)
    return x1, y1, x2, y2


def _is_large_enough(region: np.ndarray, min_side_px: int) -> bool:
    height, width = region.shape[:2]
    return max(height, width) >= min_side_px


def crop_with_padding(
    image: np.ndarray, box: BoundingBox, padding_ratio: float, min_side_px: int
) -> np.ndarray:
    """Memotong region sesuai box dengan padding, menolak hasil yang terlalu kecil.

    Args:
        image: Gambar asal, hasil `cv2.imread`.
        box: Box wajah, atau bbox upper/lower body hasil `derive_body_part_bbox`.
        padding_ratio: Rasio padding simetris di semua sisi box.
        min_side_px: Sisi terpanjang minimum hasil crop yang masih diterima.

    Returns:
        Region gambar hasil crop.

    Raises:
        CropTooSmallError: Kalau sisi terpanjang hasil crop di bawah `min_side_px`.
    """
    x1, y1, x2, y2 = _apply_padding(box, image.shape[:2], padding_ratio)
    region = image[y1:y2, x1:x2]
    if not _is_large_enough(region, min_side_px):
        raise CropTooSmallError(
            f"Crop {max(region.shape[:2])}px di bawah ambang {min_side_px}px"
        )
    return region


def save_crop(region: np.ndarray, output_path: Path) -> None:
    """Menyimpan hasil crop ke berkas.

    Args:
        region: Region hasil `crop_with_padding`.
        output_path: Path tujuan penyimpanan.

    Raises:
        CropSaveError: Kalau berkas gagal ditulis.
    """
    try:
        output_path.parent.mkdir(parents=True, exist_ok=True)
        ok = cv2.imwrite(str(output_path), region)
    except OSError as e:
        logger.error(f"Crop gagal disimpan ke {output_path}: {e}", exc_info=True)
        raise CropSaveError(f"Crop gagal disimpan ke {output_path}: {e}") from e
    if not ok:
        raise CropSaveError(f"Crop gagal disimpan ke {output_path}")


def build_crop_filename(source_path: Path, suffix: str) -> str:
    """Membentuk nama file hasil crop dari nama file asal, agar re-run menimpa (K15).

    Args:
        source_path: Path gambar asal.
        suffix: Jenis crop ("face", "upper", atau "lower").

    Returns:
        Nama file hasil crop, mis. "foto1_face.jpg".
    """
    return f"{source_path.stem}_{suffix}.jpg"

In [ ]:
# Runner "define -> demonstrate": menerapkan crop_with_padding pada wajah dan bbox
# upper/lower dari runner sebelumnya (_runner_faces, _runner_body_part_boxes), lalu
# menampilkan hasil crop-nya. Sengaja tidak dieksekusi otomatis saat pemeriksaan kode
# (lihat README.md).
_runner_face_crops = []
for _runner_path, _runner_image, _runner_boxes in zip(
    _runner_image_paths, _runner_images, _runner_faces
):
    try:
        _runner_box = validate_single_face(_runner_boxes)
    except (NoFaceDetectedError, MultipleFaceDetectedError) as e:
        print(f"{_runner_path.name}: dilewati, {e}")
        continue
    try:
        _runner_region = crop_with_padding(_runner_image, _runner_box, PADDING_RATIO, MIN_SIDE_PX)
    except CropTooSmallError as e:
        print(f"{_runner_path.name}: dilewati, {e}")
        continue
    print(f"{build_crop_filename(_runner_path, 'face')}: {_runner_region.shape[1]}x{_runner_region.shape[0]}px")
    _runner_face_crops.append(_runner_region)

_runner_body_part_crops = []
for _runner_path, _runner_group_name, _runner_box in _runner_body_part_boxes:
    _runner_image = next(
        image for path, image in zip(_runner_image_paths, _runner_images) if path == _runner_path
    )
    try:
        _runner_region = crop_with_padding(_runner_image, _runner_box, PADDING_RATIO, MIN_SIDE_PX)
    except CropTooSmallError as e:
        print(f"{_runner_path.name} [{_runner_group_name}]: dilewati, {e}")
        continue
    print(
        f"{build_crop_filename(_runner_path, _runner_group_name)}: "
        f"{_runner_region.shape[1]}x{_runner_region.shape[0]}px"
    )
    _runner_body_part_crops.append(_runner_region)

_runner_all_crops = _runner_face_crops + _runner_body_part_crops
if _runner_all_crops:
    _runner_figure, _runner_axes = plt.subplots(
        1, len(_runner_all_crops), figsize=(4 * len(_runner_all_crops), 6), squeeze=False
    )
    for _runner_axis, _runner_region in zip(_runner_axes[0], _runner_all_crops):
        _runner_axis.imshow(cv2.cvtColor(_runner_region, cv2.COLOR_BGR2RGB))
        _runner_axis.axis("off")
    plt.show()

## 8. Orkestrasi per identitas (K12)

In [ ]:
class RegionOutcome(Enum):
    """Kategori hasil pemrosesan satu jenis crop (wajah, upper, atau lower body)."""

    CROPPED = "cropped"
    SKIPPED_NO_DETECTION = "skipped_no_detection"
    SKIPPED_MULTIPLE_DETECTION = "skipped_multiple_detection"
    SKIPPED_INSUFFICIENT_KEYPOINTS = "skipped_insufficient_keypoints"
    SKIPPED_TOO_SMALL = "skipped_too_small"
    SKIPPED_UNREADABLE = "skipped_unreadable"


@dataclass
class RegionSummary:
    """Ringkasan hasil pemrosesan satu jenis crop untuk satu identitas."""

    cropped: int = 0
    skipped_no_detection: int = 0
    skipped_multiple_detection: int = 0
    skipped_insufficient_keypoints: int = 0
    skipped_too_small: int = 0
    skipped_unreadable: int = 0

    def add(self, outcome: RegionOutcome) -> None:
        """Menambah satu hitungan sesuai kategori `outcome`.

        Args:
            outcome: Hasil pemrosesan satu gambar untuk jenis crop ini.
        """
        setattr(self, outcome.value, getattr(self, outcome.value) + 1)


def total_skipped(summary: RegionSummary) -> int:
    """Menghitung total gambar yang dilewati pada satu ringkasan jenis crop.

    Args:
        summary: Ringkasan hasil salah satu field `ProcessSummary`.

    Returns:
        Jumlah gambar yang dilewati dari semua kategori.
    """
    return (
        summary.skipped_no_detection
        + summary.skipped_multiple_detection
        + summary.skipped_insufficient_keypoints
        + summary.skipped_too_small
        + summary.skipped_unreadable
    )


@dataclass
class ProcessSummary:
    """Ringkasan hasil pemrosesan satu identitas, per jenis crop (K12)."""

    face: RegionSummary = field(default_factory=RegionSummary)
    upper_body: RegionSummary = field(default_factory=RegionSummary)
    lower_body: RegionSummary = field(default_factory=RegionSummary)


def _read_image(image_path: Path) -> np.ndarray:
    try:
        image = cv2.imread(str(image_path))
    except cv2.error as e:
        logger.error(f"Gambar tidak bisa dibaca: {image_path}: {e}", exc_info=True)
        raise ImageReadError(f"Gambar tidak bisa dibaca: {image_path}") from e
    if image is None:
        raise ImageReadError(f"Gambar tidak bisa dibaca (format tidak dikenali): {image_path}")
    return image


def _list_images(identity_dir: Path) -> list[Path]:
    return sorted(p for p in identity_dir.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)


def _crop_and_save(
    image: np.ndarray,
    box: BoundingBox,
    output_path: Path,
    padding_ratio: float,
    min_side_px: int,
) -> RegionOutcome:
    try:
        region = crop_with_padding(image, box, padding_ratio, min_side_px)
    except CropTooSmallError as e:
        logger.warning(f"Dilewati, {e}: {output_path}")
        return RegionOutcome.SKIPPED_TOO_SMALL
    save_crop(region, output_path)
    return RegionOutcome.CROPPED


def _process_face(
    image: np.ndarray,
    source_path: Path,
    output_dir: Path,
    face_analyzer: FaceAnalysis,
    padding_ratio: float,
    min_side_px: int,
) -> RegionOutcome:
    boxes = detect_faces(face_analyzer, image)
    try:
        box = validate_single_face(boxes)
    except NoFaceDetectedError:
        logger.info(f"Wajah dilewati, tidak ada wajah terdeteksi: {source_path}")
        return RegionOutcome.SKIPPED_NO_DETECTION
    except MultipleFaceDetectedError as e:
        logger.warning(f"Wajah dilewati, {e}: {source_path}")
        return RegionOutcome.SKIPPED_MULTIPLE_DETECTION
    output_path = output_dir / build_crop_filename(source_path, "face")
    return _crop_and_save(image, box, output_path, padding_ratio, min_side_px)


def _process_body_part(
    image: np.ndarray,
    source_path: Path,
    output_dir: Path,
    keypoints: np.ndarray,
    indices: set[int],
    anchor_indices: set[int],
    keypoint_conf_threshold: float,
    suffix: str,
    padding_ratio: float,
    min_side_px: int,
) -> RegionOutcome:
    try:
        box = derive_body_part_bbox(keypoints, indices, anchor_indices, keypoint_conf_threshold)
    except InsufficientKeypointsError as e:
        logger.info(f"{suffix.capitalize()} dilewati, {e}: {source_path}")
        return RegionOutcome.SKIPPED_INSUFFICIENT_KEYPOINTS
    output_path = output_dir / build_crop_filename(source_path, suffix)
    return _crop_and_save(image, box, output_path, padding_ratio, min_side_px)


def process_image(
    image_path: Path,
    face_dir: Path,
    upper_dir: Path,
    lower_dir: Path,
    face_analyzer: FaceAnalysis,
    pose_model: YOLO,
    padding_ratio: float,
    min_side_px: int,
    pose_conf_threshold: float,
    keypoint_conf_threshold: float,
) -> tuple[RegionOutcome, RegionOutcome, RegionOutcome]:
    """Memproses satu gambar untuk tiga jenis crop, independen satu sama lain (K12).

    Args:
        image_path: Path gambar hasil modul 001, dari `data/cropped/<identitas>/`.
        face_dir: Folder tujuan crop wajah untuk identitas ini.
        upper_dir: Folder tujuan crop upper body untuk identitas ini.
        lower_dir: Folder tujuan crop lower body untuk identitas ini.
        face_analyzer: FaceAnalysis hasil `load_face_analyzer`.
        pose_model: Model YOLO hasil `load_pose_detector`.
        padding_ratio: Rasio padding simetris sebelum crop.
        min_side_px: Sisi terpanjang minimum hasil crop yang masih diterima.
        pose_conf_threshold: Ambang confidence minimum deteksi orang oleh model pose.
        keypoint_conf_threshold: Ambang confidence minimum per-keypoint.

    Returns:
        Outcome (wajah, upper body, lower body) untuk gambar ini. Kegagalan pada
        satu jenis tidak menggagalkan jenis lain.
    """
    try:
        image = _read_image(image_path)
    except ImageReadError as e:
        logger.warning(str(e))
        return (
            RegionOutcome.SKIPPED_UNREADABLE,
            RegionOutcome.SKIPPED_UNREADABLE,
            RegionOutcome.SKIPPED_UNREADABLE,
        )

    face_outcome = _process_face(image, image_path, face_dir, face_analyzer, padding_ratio, min_side_px)

    people = detect_pose(pose_model, image_path, pose_conf_threshold)
    try:
        keypoints = validate_single_person_pose(people)
    except NoPersonPoseDetectedError:
        logger.info(f"Upper/lower dilewati, tidak ada orang terdeteksi pose: {image_path}")
        return face_outcome, RegionOutcome.SKIPPED_NO_DETECTION, RegionOutcome.SKIPPED_NO_DETECTION
    except MultiplePersonPoseDetectedError as e:
        logger.warning(f"Upper/lower dilewati, {e}: {image_path}")
        return face_outcome, RegionOutcome.SKIPPED_MULTIPLE_DETECTION, RegionOutcome.SKIPPED_MULTIPLE_DETECTION

    upper_outcome = _process_body_part(
        image,
        image_path,
        upper_dir,
        keypoints,
        UPPER_KEYPOINT_INDICES,
        UPPER_ANCHOR_INDICES,
        keypoint_conf_threshold,
        "upper",
        padding_ratio,
        min_side_px,
    )
    lower_outcome = _process_body_part(
        image,
        image_path,
        lower_dir,
        keypoints,
        LOWER_KEYPOINT_INDICES,
        LOWER_ANCHOR_INDICES,
        keypoint_conf_threshold,
        "lower",
        padding_ratio,
        min_side_px,
    )
    return face_outcome, upper_outcome, lower_outcome


def process_identity(
    identity_dir: Path,
    face_dir: Path,
    upper_dir: Path,
    lower_dir: Path,
    face_analyzer: FaceAnalysis,
    pose_model: YOLO,
    padding_ratio: float,
    min_side_px: int,
    pose_conf_threshold: float,
    keypoint_conf_threshold: float,
) -> ProcessSummary:
    """Menjalankan deteksi wajah/upper/lower body dan cropping untuk satu identitas.

    Args:
        identity_dir: Folder hasil crop modul 001, dari `data/cropped/<identitas>/`.
        face_dir: Folder tujuan crop wajah, `data/face/<identitas>/`.
        upper_dir: Folder tujuan crop upper body, `data/upper_body/<identitas>/`.
        lower_dir: Folder tujuan crop lower body, `data/lower_body/<identitas>/`.
        face_analyzer: FaceAnalysis hasil `load_face_analyzer`, dipakai bersama semua identitas.
        pose_model: Model YOLO hasil `load_pose_detector`, dipakai bersama semua identitas.
        padding_ratio: Rasio padding simetris sebelum crop.
        min_side_px: Sisi terpanjang minimum hasil crop yang masih diterima.
        pose_conf_threshold: Ambang confidence minimum deteksi orang oleh model pose.
        keypoint_conf_threshold: Ambang confidence minimum per-keypoint.

    Returns:
        Ringkasan jumlah crop dan gambar yang dilewati, per jenis crop.
    """
    if not identity_dir.exists():
        logger.warning(f"Folder identitas tidak ditemukan: {identity_dir}")
        return ProcessSummary()

    for output_dir in (face_dir, upper_dir, lower_dir):
        output_dir.mkdir(parents=True, exist_ok=True)

    summary = ProcessSummary()
    for image_path in _list_images(identity_dir):
        face_outcome, upper_outcome, lower_outcome = process_image(
            image_path,
            face_dir,
            upper_dir,
            lower_dir,
            face_analyzer,
            pose_model,
            padding_ratio,
            min_side_px,
            pose_conf_threshold,
            keypoint_conf_threshold,
        )
        summary.face.add(face_outcome)
        summary.upper_body.add(upper_outcome)
        summary.lower_body.add(lower_outcome)

    logger.info(
        f"[{identity_dir.name}] wajah: {summary.face.cropped} crop, "
        f"upper: {summary.upper_body.cropped} crop, lower: {summary.lower_body.cropped} crop"
    )
    return summary


def run_pipeline(identities: dict[str, Path]) -> dict[str, ProcessSummary]:
    """Menjalankan pipeline modul 002 untuk setiap identitas.

    Args:
        identities: Pemetaan nama identitas ke folder `data/cropped/<identitas>/`.

    Returns:
        Ringkasan hasil pemrosesan per identitas.
    """
    face_analyzer = load_face_analyzer(FACE_MODEL_NAME, FACE_DET_THRESH)
    pose_model = load_pose_detector(POSE_MODEL_WEIGHTS)

    summaries: dict[str, ProcessSummary] = {}
    for identity_name, identity_dir in identities.items():
        summaries[identity_name] = process_identity(
            identity_dir,
            FACE_DIR / identity_name,
            UPPER_BODY_DIR / identity_name,
            LOWER_BODY_DIR / identity_name,
            face_analyzer,
            pose_model,
            PADDING_RATIO,
            MIN_SIDE_PX,
            POSE_CONF_THRESHOLD,
            KEYPOINT_CONF_THRESHOLD,
        )
    return summaries

In [ ]:
# Runner "define -> demonstrate": menjalankan pipeline untuk semua folder di
# data/cropped/ (hasil modul 001) dan menampilkan ringkasan DISPLAY_LIMIT identitas
# pertama. Jalankan notebook 01 sampai selesai lebih dulu supaya data/cropped/
# terisi. Sel ini mengunduh bobot buffalo_sc dan yolov8n-pose.pt (butuh koneksi
# internet sekali) dan memanggil model sungguhan — sengaja tidak dijalankan otomatis
# saat pemeriksaan kode (lihat README.md).
IDENTITIES = {
    identity_dir.name: identity_dir
    for identity_dir in sorted(CROPPED_DIR.iterdir())
    if identity_dir.is_dir()
} if CROPPED_DIR.exists() else {}

summaries = run_pipeline(IDENTITIES)
for identity_name, summary in islice(summaries.items(), DISPLAY_LIMIT):
    print(
        f"{identity_name}: wajah {summary.face.cropped} crop/{total_skipped(summary.face)} dilewati, "
        f"upper {summary.upper_body.cropped} crop/{total_skipped(summary.upper_body)} dilewati, "
        f"lower {summary.lower_body.cropped} crop/{total_skipped(summary.lower_body)} dilewati"
    )